# Milestone 4 lab, notebook 8 of 8: The production version

## Where we are

| Notebook | What you do |
|---|---|
| 1. A rule lives in dbt | Move a check out of a notebook and into a file that dbt runs |
| 2. How serious is it | Tell dbt which failures stop the night. Keep the rows a test returns |
| 3. Test the SQL, not the data | Check your own query on four rows typed by hand |
| 4. A gate, and where it stands | Run one rule in three places, and see what each place can see |
| 5. Warn, block or set aside | Decide what a failure does. Then block a real run in Airflow |
| 6. Quarantine | Set the bad rows aside with a reason, and let the good rows go on |
| 7. Test the tests | Plant a fault, and prove its gate catches it. One check in four tools |
| **8. The production version** (you are here) | Run the whole week through Airflow. See what is still wrong |

**You need:** the warehouse and Airflow running: `docker compose up -d --build`.

**About five minutes of reading, and four minutes of waiting** while Airflow runs seven nights.

In [1]:
import subprocess
import sys

import pandas as pd
from lab_helpers import (
    MILESTONE_DIR, WEEK, bonus_report, can_be_paid, load_result, reset_warehouse, save_result,
    show_report, sql,
)

import airflow_tools  # lab_helpers put this milestone's folder on the path, so this is found

airflow_tools.wait_until_ready()

## Step 1: From a lab to a week at 02:00

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, and what happens when it fails.

In the lab you ran dbt by hand, one step at a time. In production nobody does. Airflow starts eight tasks every night, and three of them are gates.

So run the bonus week the way it really ran. Empty the warehouse of this milestone, and let Airflow do seven nights, Monday to Sunday. Each night takes about half a minute.

After every night, ask the Monday question: how many stores could be paid right now?

In [2]:
reset_warehouse()

nights = []
for night in WEEK:
    states = airflow_tools.run_night(night.isoformat())
    nights.append(
        {
            "night": f"{night:%a %m-%d}",
            "tasks": len(states),
            "succeeded": sum(s["state"] == "success" for s in states),
            "stores that can be paid": can_be_paid(night),
        }
    )
pd.DataFrame(nights)

The warehouse of this milestone is empty.


,night,tasks,succeeded,stores that can be paid
0,Mon 08-24,8,8,5
1,Tue 08-25,8,8,4
2,Wed 08-26,8,8,4
3,Thu 08-27,8,8,2
4,Fri 08-28,8,8,3
5,Sat 08-29,8,8,2
6,Sun 08-30,8,8,1


Seven nights, eight green tasks each. No gate blocked: nothing in this week breaks a reference table.

And the last column tells the week. Five stores on Monday, then four, four, two. On Friday it goes **up** to three: store `S06`'s late file arrived, and its Thursday became right. Then two, and on Sunday night one.

## Step 2: The report on Monday morning

In [3]:
show_report()

Lotus Retails. Store managers' bonus, week of Mon 2026-08-24
Net sales from Mon 2026-08-24 to Sun 2026-08-30

  Store                           Manager              Net sales  Bonus (1%)   Pay?
  S01  Lotus Ashby High Street    Marta Lindqvist      $5,153.25      $51.53   yes
  S02  Lotus Ashby Market Square  Dana Okafor          $5,724.68           -   DO NOT PAY YET
         Sun 08-30: 4 rows set aside ($23.95): Nothing is sold while the store is closed
  S03  Lotus Calder Bridge        Tomas Reyes          $5,097.11           -   DO NOT PAY YET
         Mon 08-24: 1 row set aside ($9.49): A line adds up; A refund gives money back
  S04  Lotus Calder Station       Ines Marchetti       $4,585.57           -   DO NOT PAY YET
         Tue 08-25: no sales file arrived. The tills counted $587.42
  S05  Lotus Fenwick Green        Samir Haddad         $3,591.53           -   DO NOT PAY YET
         Thu 08-27: 1 row set aside ($1,522.20): A weight is a believable weight
  S06  Lotus Fenwick 

Compare it with the report of milestone 1: six stores, six numbers, "Checks: 5 of 5 passed", and every number wrong.

This one is not all good news. It is all **true**. One store can be paid. Five cannot, and each line says the day, the rows, the money and the rule.

Save it. A test holds the production code against what you see here.

In [4]:
ready = bonus_report()
aside = sql("""
    SELECT 'sales_lines' AS in_table, count(DISTINCT line_id) AS rows FROM quarantine.sales_lines
    UNION ALL SELECT 'raw_sales_lines', count(*) FROM quarantine.raw_sales_lines
    UNION ALL SELECT 'supplier_prices', count(*) FROM quarantine.supplier_prices
    UNION ALL SELECT 'stock_counts', count(*) FROM quarantine.stock_counts
    UNION ALL SELECT 'members', count(DISTINCT member_id) FROM quarantine.members
""")
refused = sql("SELECT file_name FROM ops.refused_files ORDER BY file_name")

save_result(
    "answer_key_gates",
    {
        "night": "2026-08-30",
        "can_be_paid": [s["store_id"] for s in ready["stores"] if s["pay"]],
        "why_not": {s["store_id"]: s["why_not"] for s in ready["stores"] if not s["pay"]},
        "rows_set_aside": {row["in_table"]: int(row["rows"]) for _, row in aside.iterrows()},
        "files_refused": refused["file_name"].tolist(),
    },
)
aside

Saved notebooks/results/answer_key_gates.json


,in_table,rows
0,sales_lines,177
1,raw_sales_lines,3
2,supplier_prices,0
3,stock_counts,0
4,members,1


## Step 3: The tests

`tests/test_gates.py` does what you did in notebooks 5 to 7, on every change of the code: it runs the week night by night, checks every planted case at its gate, and plants the faults that block. It takes about a minute.

In [5]:
done = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_gates.py", "-v", "--no-header", "--color=no",
     "-p", "no:cacheprovider"],
    cwd=MILESTONE_DIR, capture_output=True, text=True,
)
names = []
for line in done.stdout.splitlines():
    if "::" in line and "PASSED" in line:
        name = line.split("::")[1].split("[")[0].split(" ")[0]
        if name not in names:
            names.append(name)
print("\n".join(names))
print(done.stdout.strip().splitlines()[-1])

test_four_clean_weeks_pass_every_gate_and_every_store_can_be_paid
test_nothing_that_is_not_wrong_is_touched
test_the_week_night_by_night
test_on_monday_one_store_can_be_paid_and_five_cannot_each_with_its_reason
test_what_is_published_is_the_truth_less_what_was_set_aside
test_case_1_the_refund_is_set_aside_on_the_first_night
test_case_2_and_9_extra_copies_are_removed_and_the_day_is_right_again
test_case_3_5_and_6_are_set_aside_with_their_reason
test_case_4_is_a_warning_and_the_store_can_still_be_paid_for_that_day
test_case_7_no_file_and_case_8_a_late_file
test_case_10_and_11_two_supplier_files_are_refused_and_the_last_good_cost_is_used
test_case_12_a_missing_count_is_not_read_as_an_empty_shelf
test_case_13_marketing_gets_one_member_and_no_personal_data_is_set_aside
test_running_a_night_again_changes_nothing
test_a_broken_reference_table_blocks_and_the_outputs_keep_their_last_good_numbers
test_two_prices_on_one_day_block
test_a_difference_nobody_can_explain_blocks_at_the_publish_gate
tes

In [6]:
saved = load_result("answer_key_gates")
print("Can be paid:", ", ".join(saved["can_be_paid"]))
print("Do not pay yet:", ", ".join(saved["why_not"]))
print("Rows set aside:", saved["rows_set_aside"])
print("Files refused:", len(saved["files_refused"]))

Can be paid: S01
Do not pay yet: S02, S03, S04, S05, S06
Rows set aside: {'sales_lines': 177, 'raw_sales_lines': 3, 'supplier_prices': 0, 'stock_counts': 0, 'members': 1}
Files refused: 2


## Step 4: What is still wrong

Look at the report once more, and find **Ines Marchetti**, store `S04`.

Her Tuesday file never arrived. The publish gate knew it at 02:00 on Wednesday morning. It wrote a correct row into a table, and a warning into a table.

And then? Nobody was told. Not Ines, who could have sent the file again on Wednesday. Not the person who runs the pipeline. The warning sat in `ops.gate_results` on Wednesday, Thursday, Friday, Saturday and Sunday.

In [7]:
sql("""
    SELECT run_day, bad_rows AS store_days_not_ready, title
    FROM ops.gate_results
    WHERE test_name = 'store_is_not_ready_to_pay'
    ORDER BY run_day
""")

,run_day,store_days_not_ready,title
0,2026-08-24,1,Every open store has a day we can pay from
1,2026-08-25,2,Every open store has a day we can pay from
2,2026-08-26,2,Every open store has a day we can pay from
3,2026-08-27,4,Every open store has a day we can pay from
4,2026-08-28,3,Every open store has a day we can pay from
5,2026-08-29,4,Every open store has a day we can pay from
6,2026-08-30,5,Every open store has a day we can pay from


Seven nights, seven rows, and no person read one of them. On Monday the head of retail operations opens the report, and Ines hears for the first time that she cannot be paid, for a file that has been missing for six days.

The report is right. It is also six days late for the one person who could have fixed it.

> **A gate fails at 02:00, and nobody knows until a store manager complains.**

Milestone 5 tells the right person, at the right time, and then repairs the week.

## Before you go on

- Open [milestone_4_flow.html](../milestone_4_flow.html) in a browser. Watch a bad row stop at a gate, and a good row pass.
- Open [milestone_4_recap.html](../milestone_4_recap.html) for a look back, and three questions.
- Stop what you started: `docker compose down`, from the project folder. Your data is kept.